In [0]:
import requests
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
years = [2016, 2017, 2018]
holiday_raw = []
for year in years:
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/BR"
    response = requests.get(url)
    print(f"Loading {year}: {response.status_code}")
    if response.status_code == 200:
        holidays = response.json()
        for h in holidays:
            holiday_raw.append(
                (
                    h.get("date"),
                    h.get("localName"),
                    h.get("name"),
                    h.get("countryCode"),
                    h.get("fixed"),
                    h.get("global"),
                    str(h.get("counties")),
                    str(h.get("launchYear")),
                    str(h.get("types")),
                    year,
                    datetime.now()
                )
            )

    else:
        print(response.text)

In [0]:
columns = [
    "holiday_date",
    "holiday_local_name",
    "holiday_name",
    "country_code",
    "is_fixed",
    "is_global",
    "counties",
    "launch_year",
    "holiday_types",
    "year",
    "ingestion_time"
]


df_holiday = spark.createDataFrame(
    holiday_raw,
    columns
)

In [0]:
df_holiday.printSchema()

In [0]:
df_holiday.display()

In [0]:
(
    df_holiday
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "olist_catalog.api_to_bronze.brazil_holidays_raw"
    )
)

In [0]:
spark.sql("""
SELECT *
FROM olist_catalog.api_to_bronze.brazil_holidays_raw
ORDER BY holiday_date
""").display()